# Day 2 · Module 3 — From a Variant Score to a Patient's Risk (BRCA1 Triage)

**Driving question:** a model can rank a *variant* as damaging with >90% separation — why is that not the same as telling a patient they will get sick?

**Objective:** run the zero-shot variant scorer end-to-end on BRCA1, reproduce the benign-vs-pathogenic separation with the frontier-model table, build a triage worklist — then reckon honestly with the gap between a *variant effect* and a *patient's disease risk* (penetrance, polygenicity, dual-use).

> **Instructor solution** · kernel **AImed**.

### References
- **Day-2 M2 method:** a variant's score is `delta = LL(alt_window) − LL(ref_window)` — how much *less natural* the genome looks with the variant swapped in. More negative = more disruptive. `AUROC(is_lof, −delta)` is the Day-1 ranking muscle, now on genomic scores.
- **The measured numbers (errata):** HyenaDNA-small is **≈ chance on BRCA1 (AUROC ~0.46)**; the headline separation is the **Evo 2 7B** table (**AUROC 0.877**), generated once on big hardware and shipped to you. Same method, different-size model.
- **ClinVar / VUS:** every variant carries a clinical call — *Benign, Pathogenic*, or **Uncertain significance (VUS)** — the unknowns a lab wants triaged.
- *Today's Goal:* what is the difference between *'this variant is damaging'* and *'this patient is at risk'*?

In [ ]:
# --- Environment setup (run me first; you don't need to edit this) ---
# Finds the shared course 'scripts/' folder and puts it on Python's import path so the
# later cells can do `import clinical_risk_console` and `from common import metrics`.
import sys, pathlib, os
cands = [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]   # current dir + all parent dirs
if os.environ.get('PROJECT_DIR'):                            # + the shared course dir if it is set
    cands.append(pathlib.Path(os.environ['PROJECT_DIR']))
scripts = next((b/'scripts' for b in cands                   # first candidate that has scripts/common/
                if (b/'scripts'/'common'/'aimed_config.py').exists()), None)
# if this assert fires, your kernel isn't seeing the course tree -> relaunch the AImed kernel
assert scripts, 'run from inside the course tree, or set PROJECT_DIR to the shared course dir'
sys.path.insert(0, str(scripts))                             # make those scripts importable below
from common import aimed_config as cfg                       # shared paths + cache configuration
cfg.setup_caches()                                           # point HF/torch caches at shared storage
print('environment OK  ->', cfg.PROJECT_DIR)

## Submodule 3a — Does the score work, and on what?
## Guided hands-on

### Background — what a *zero-shot variant-effect predictor* is

**The model.** A DNA language model (here **HyenaDNA-small**, the one you can run on a laptop CPU; or the much larger **Evo 2 7B**) was trained on raw genomes to do one thing: predict the next DNA base, over and over, the way a text model predicts the next word. It saw **zero clinical labels** — nobody ever told it which variants cause disease.

**The trick — score a variant without ever training on variants.** A gene like **BRCA1** is a stretch of DNA. A *single-nucleotide variant (SNV)* changes one letter (e.g. an `A` becomes a `T`) at one position. We show the model two versions of an ~8 kb window of chromosome 17: the **reference** (what most people have) and the **alt** (with the one letter swapped). Each window gets a log-likelihood — *how natural / unsurprising this sequence looks to the model.* The signal is the difference:

> `delta = LL(alt_window) − LL(ref_window)`  — a **more negative delta = the edit made the genome look weirder = more likely to break the protein.**

**Why this can possibly work.** The implicit supervision is **evolution.** Sequences that break important genes got selected *out* of the genomes the model trained on, so the model learned they are 'unnatural.' Pathogenicity leaks in through that door — no clinical labels required. This is **zero-shot**: predicting a task it was never explicitly trained for.

**The catch you'll measure today:** the small model (HyenaDNA ≈ chance) barely separates pathogenic from benign; the larger model separates them cleanly (Evo 2, AUROC 0.877). **Same method — the difference is scale.** And even a perfect scorer only scores the *variant*, not the *patient* (that's Submodule 3b).

In [ ]:
# Guided hands-on — score the same gene with two models, the SAME method.
# run_vep(use_precomputed=True) reads the precomputed HyenaDNA delta table (no GPU, no network).
# evo2_auroc() reads the frontier Evo 2 7B table the instructor generated once on big hardware.
from vep_scorer import run_vep, evo2_auroc, evo2_scored_table
import brca1_triage as BT

# --- the SMALL model you can actually run: HyenaDNA-small on your own BRCA1 rows ---
df, hyena_auroc, info = run_vep(n=400, use_precomputed=True, log=lambda *a: None)
print(f'HyenaDNA-small  AUROC = {hyena_auroc:.3f}   (~chance)')
print(f'  source = {info["source"]}')

# --- the FRONTIER model, consumed as a table (identical method, bigger model) ---
evo2 = evo2_auroc()
print(f'Evo 2 7B        AUROC = {evo2:.3f}   (separates pathogenic from benign)')
print(f'  delta(AUROC)        = {evo2 - hyena_auroc:+.3f}  -> same pipeline, scale is what was missing.')

### Key terms
- **`delta` (the score):** `LL(alt) − LL(ref)`. Negative = the edit made the sequence look less natural = more likely loss-of-function. We rank by **`−delta`** so 'higher = more pathogenic.'
- **`is_lof`:** the ground-truth label (1 = loss-of-function) from a deep mutational scan — our 'right answers' for measuring AUROC.
- **AUROC:** chance a random truly-LoF variant outranks a random benign one. 0.5 = coin flip, 1.0 = perfect.
- **VUS — Variant of Uncertain Significance:** ClinVar has *no confident call*. These are the variants a lab actually wants ranked — the worklist you'll build below.
- **Zero-shot:** the model was never trained on the pathogenicity task; the signal comes from evolution baked into the training genomes.

In [ ]:
# Demo — reproduce the >90%-style separation on your own rows, with the frontier model.
# evo2_scored_table() returns the full BRCA1 dataframe with an Evo 2 'delta' column merged on
# (by pos/ref/alt) -- so we can look at the ACTUAL clinical groups, not just one AUROC number.
import numpy as np
from common import metrics as M
e = evo2_scored_table()                      # full BRCA1 table + Evo 2 'delta'
clin = e[e['clinvar'].isin(['Pathogenic', 'Likely pathogenic', 'Benign', 'Likely benign'])].copy()
clin['is_path'] = clin['clinvar'].isin(['Pathogenic', 'Likely pathogenic']).astype(int)
ben  = clin.loc[clin.is_path == 0, 'delta']
path = clin.loc[clin.is_path == 1, 'delta']
print(f'ClinVar-labeled variants: {len(clin)}  ({len(path)} pathogenic, {len(ben)} benign)')
print(f'  mean delta  benign = {ben.mean():+.4f}   pathogenic = {path.mean():+.4f}   (pathogenic is LOWER)')
print(f'  AUROC (pathogenic vs benign) = {M.auroc(clin.is_path, -clin.delta):.3f}   <- this IS the >90% separation headline')
# a STRICTER, different metric: accuracy under ONE fixed cutoff at the midpoint of the class means
thr = (ben.mean() + path.mean()) / 2
sep = ((path < thr).sum() + (ben >= thr).sum()) / len(clin)
print(f'  single fixed-cutoff accuracy = {100*sep:.0f}% correctly sorted (one threshold, no tuning).')
print('--> The frontier model, with ZERO clinical labels, sorts known-pathogenic from known-benign. THIS is the headline.')

In [ ]:
# YOUR TURN — build the artifact a genetics lab actually wants: a ranked VUS worklist.
# A VUS has NO confident clinical call. We rank them by the frontier model's delta, most
# disruptive first, to propose which to investigate first. Fill each ...  (solution has answers)
import numpy as np
from common import metrics as M

# (a) Which model's deltas do you trust to rank the unknowns? You measured both AUROCs above.
#     Pick the dataframe whose deltas are actually predictive. (hint: not the ~chance one.)
ranking_df = e        # <-- choose: `df` (HyenaDNA ~chance) or `e` (Evo 2, 0.877)

# (b) Rank the Variants of Uncertain Significance by delta, most disruptive (lowest delta) first.
#     BT.vus_triage(dataframe, top=N) -> a table sorted ascending by delta.
vus = BT.vus_triage(ranking_df, top=10)
print(f'Top {len(vus)} VUS to prioritise for follow-up (most disruptive first):')
print(vus.to_string(index=False))

# (c) Sanity-check your choice: compute the AUROC of YOUR ranking_df's deltas vs is_lof.
#     If it is near 0.5 you are ranking the worklist by noise -- pick the other dataframe.
my_auroc = M.auroc(ranking_df['is_lof'], -ranking_df['delta'])          # <-- M.auroc(ranking_df['is_lof'], -ranking_df['delta'])
print(f'\nAUROC of the deltas you ranked by = {my_auroc:.3f}  ', end='')
print('(near 0.5 means your worklist is noise!)' if my_auroc < 0.6 else '(predictive -> a defensible worklist)')

In [ ]:
# --- Static visual: the two delta distributions the AUROC summarises --------------------
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
fig, ax = plt.subplots(figsize=(7, 4.2))
bins = np.linspace(min(clin.delta.min(), -0.03), max(clin.delta.max(), 0.01), 40)
ax.hist(ben,  bins=bins, alpha=0.6, color='C0', label=f'Benign (n={len(ben)})')
ax.hist(path, bins=bins, alpha=0.6, color='C3', label=f'Pathogenic (n={len(path)})')
ax.axvline(thr, color='k', ls='--', lw=1, label=f'one cutoff ({thr:+.4f})')
ax.set(xlabel='Evo 2 delta  (more negative = more disruptive)', ylabel='count',
       title='Evo 2 separates pathogenic from benign on BRCA1 (zero clinical labels)')
ax.legend()
plt.tight_layout(); plt.show()
print('Read it: the red (pathogenic) mass sits LEFT of the blue (benign) mass. The overlap is the error rate.')

### Your task — which model gets to touch the worklist, and why?
Using the AUROCs you measured, fill in the **`DECISION_3A`** dict. There is no single right answer — what matters is the reasoning (argue it in the discussion). Be specific: cite the two AUROC numbers, and say what a clinician would conclude if you handed them a worklist ranked by the ~chance model.

In [ ]:
# A defended choice (what matters is the reasoning; cite both AUROCs).
DECISION_3A = {
    'model_for_worklist': 'Evo 2 7B',
    'why': ('HyenaDNA-small ranks BRCA1 at ~chance (AUROC ~0.46) while Evo 2 7B reaches 0.877, so only '
            'the frontier deltas carry signal worth triaging by'),
    'risk_if_wrong_model': ('triaging by the ~chance ranking sends the lab to investigate essentially '
                            'random variants, wasting scarce review while real LoF variants wait'),
    'same_method': False,   # the method (delta = LL(alt)-LL(ref)) is identical; only the model differs
}
print(DECISION_3A)

In [ ]:
# CHECKPOINT (non-blocking) — stuck? This VISUALISES which model you can trust.
%matplotlib inline
import matplotlib.pyplot as plt, numpy as np
from vep_scorer import run_vep, evo2_scored_table
from common import metrics as M
df_h, hyena_auroc, _ = run_vep(n=400, use_precomputed=True, log=lambda *a: None)
e = evo2_scored_table()
fig, (axL, axR) = plt.subplots(1, 2, figsize=(11, 4))
# LEFT: ROC-like summary -- the two AUROC bars
axL.bar(['HyenaDNA\n(small)', 'Evo 2 7B\n(frontier)'],
        [hyena_auroc, M.auroc(e.is_lof, -e.delta)], color=['0.6', 'C2'])
axL.axhline(0.5, color='r', ls=':', label='chance (0.5)')
axL.set(ylim=(0, 1), ylabel='AUROC (is_lof vs -delta)', title='Same method, different scale')
axL.legend()
# RIGHT: delta distributions for the frontier model by clinical label
clin = e[e.clinvar.isin(['Pathogenic','Likely pathogenic','Benign','Likely benign'])].copy()
clin['p'] = clin.clinvar.isin(['Pathogenic','Likely pathogenic']).astype(int)
axR.hist(clin.loc[clin.p==0,'delta'], bins=30, alpha=0.6, color='C0', label='Benign')
axR.hist(clin.loc[clin.p==1,'delta'], bins=30, alpha=0.6, color='C3', label='Pathogenic')
axR.set(xlabel='Evo 2 delta', title='Frontier model: pathogenic sits left'); axR.legend()
plt.tight_layout(); plt.show()
print('Left bar near 0.5 = ranking by noise; the tall green bar is the model you can trust to triage VUS.')

**How to read the checkpoint plot.** *Left — the two AUROC bars.* A model at chance (0.5, the red line) produces a worthless ranking. The grey bar (HyenaDNA-small) is barely above that line; the green bar (Evo 2) is the model whose rankings you'd trust. **Rank your VUS worklist by the green-bar model, never the grey one** — that is your `model_for_worklist`. *Right — the frontier delta distributions.* The red (pathogenic) variants pile up to the **left** (more negative delta = more disruptive) and the blue (benign) to the right; the small overlap is exactly the ~12% the model gets wrong. Both models ran the **identical method** — the only difference is how much DNA each one read.

## Submodule 3b — Why a damaging variant is not a diagnosis
*A perfect variant score still is not a patient's risk. Here is the gap — and the dual-use question.*

### Background — the four things that stand between a score and a patient

You just built a worklist that ranks *variants*. A clinician asks a different question: *will this person get sick?* Four gaps separate the two — name each:

1. **Penetrance.** A pathogenic BRCA1 variant **raises** lifetime breast-cancer risk to roughly **60–70%**, not 100%. The variant is a loaded die, not a verdict — many carriers never develop disease. The model's `delta` says *the protein is likely broken*; it says **nothing** about *that 60%.*
2. **Zygosity & gene context.** One damaged copy vs two; **modifier genes** and environment shift the same variant's risk up or down. The score sees one position in isolation.
3. **Polygenic reality.** BRCA1 is a rare, single, **high-impact** gene. Most common disease (heart disease, diabetes, height) is **thousands of tiny-effect variants** summed into a *polygenic risk score* — a fundamentally different model. Single-variant scoring does not even apply.
4. **Score-to-clinic translation.** Before a variant changes a chart it needs **ACMG review**: orthogonal evidence, family data, functional assays. A false 'pathogenic' call costs a real family unnecessary surgery or panic. **The model is a *prior*, not a diagnosis.**

**The dual-use turn.** Today's scorers are *predictive* — they read DNA and rank it. Frontier genomic models like Evo 2 are also *generative* — they can **write** novel DNA. That is the day's safety headline: a model that designs sequences carries **biosafety / dual-use** risk a read-only clinical classifier does not. Evo 2's developers deliberately **excluded eukaryote-infecting viruses** from training and red-teamed generative outputs. You'll vote on why that distinction matters.

In [ ]:
# Guided compute — turn ONE top-ranked variant into a (rough) PATIENT risk, to feel the gap.
# The model gave us a delta. A clinician needs a probability of disease. Watch how much
# extra, NON-model information that takes.
import numpy as np
from vep_scorer import evo2_scored_table
e = evo2_scored_table()
top_vus = e[e.clinvar == 'Uncertain significance'].sort_values('delta').iloc[0]
print('Most-disruptive VUS the model flagged:')
print(f"  pos_hg19={int(top_vus.pos_hg19)}  {top_vus.reference}>{top_vus.alt}  "
      f"consequence={top_vus.get('consequence','?')}  delta={top_vus.delta:+.4f}")
print(f"  the model's output is a SCORE ({top_vus.delta:+.4f}) -- not a probability of cancer.\n")

# Penetrance is an EXTERNAL clinical fact, NOT from the model. Published BRCA1 penetrance ~0.60.
PENETRANCE = 0.60          # lifetime risk for a CONFIRMED pathogenic carrier (from cohort studies)
BASELINE   = 0.12          # population lifetime breast-cancer risk for comparison
print(f'IF this VUS were confirmed pathogenic, lifetime risk ~= {PENETRANCE:.0%} '
      f'(vs {BASELINE:.0%} baseline) -- a {PENETRANCE/BASELINE:.1f}x relative risk.')
print('Notice: penetrance, baseline, and "IF confirmed" all came from OUTSIDE the model.')

In [ ]:
# YOUR TURN — show that the SAME score implies very different patient risks. Fill each ...
# Penetrance estimates vary across studies; the model's delta does not move. Compute the
# carrier's lifetime risk under an OPTIMISTIC and a PESSIMISTIC penetrance and compare.
p_low  = 0.45        # <-- optimistic lifetime penetrance for a pathogenic BRCA1 carrier, e.g. 0.45
p_high = 0.72        # <-- pessimistic, e.g. 0.72
spread = p_high - p_low        # <-- p_high - p_low : the uncertainty the MODEL cannot resolve
print(f'Same variant, same delta. Lifetime risk ranges {p_low:.0%} .. {p_high:.0%}'
      f'  (a {spread:.0%}-point spread the model is silent on).')
# And the gate before ANY of this reaches a chart:
print('Before clinic: ACMG review + orthogonal evidence must first reclassify the VUS as pathogenic.')
print('Lesson: the model gives a PRIOR on the variant; penetrance + ACMG give the patient story.')

### Your task — the dual-use safety call (⚠️ debrief)
Fill in **`DECISION_3B`**. This is the ethics moment of Day 2. No single right answer — you defend it in the debrief. Anchor on the *predictive vs generative* distinction and the *prior vs verdict* distinction you just computed.

In [ ]:
# A defended safety call (argued in the debrief).
DECISION_3B = {
    'before_chart': ('an ACMG/AMP multi-criteria review plus orthogonal evidence must reclassify the VUS '
                     'as pathogenic before anything reaches a patient chart'),
    'generative_risk': ('a generative genomic model can be prompted to DESIGN novel sequences (e.g. fitter '
                        'pathogens) -- a dual-use capability a purely predictive scorer lacks'),
    'responsible_release': 'exclude eukaryote-infecting viruses from training + red-team before release',
    'biggest_gap': ('penetrance -- even a confirmed-pathogenic BRCA1 variant gives only ~45-72% lifetime '
                    'risk, so a single delta cannot become a personal probability'),
}
print(DECISION_3B)

In [ ]:
# CHECKPOINT (non-blocking) — VISUALISE the score-vs-risk gap so you can read the answer off.
%matplotlib inline
import matplotlib.pyplot as plt, numpy as np
BASELINE = 0.12
labels = ['population\nbaseline', 'carrier\n(optimistic 45%)', 'carrier\n(pessimistic 72%)']
risks  = [BASELINE, 0.45, 0.72]
fig, ax = plt.subplots(figsize=(7, 4.2))
bars = ax.bar(labels, risks, color=['0.6', 'C1', 'C3'])
ax.axhspan(0.45, 0.72, color='C1', alpha=0.12)
ax.annotate('the model\ncannot resolve\nthis spread', xy=(2, 0.58), xytext=(0.6, 0.85),
            ha='center', arrowprops=dict(arrowstyle='->'))
ax.set(ylim=(0, 1), ylabel='lifetime breast-cancer risk', title='One variant score -> a RANGE of patient risk')
for b, r in zip(bars, risks):
    ax.text(b.get_x()+b.get_width()/2, r+0.02, f'{r:.0%}', ha='center')
plt.tight_layout(); plt.show()
print('Read it: the model outputs ONE delta, but the patient risk is a band (penetrance) on top of a baseline.')

**How to read the checkpoint plot.** The grey bar is everyone's baseline risk; the orange-to-red band is where a *confirmed* carrier lands — and the model is **silent about where in that band** any individual falls. The shaded zone is the gap your `biggest_gap` answer names. The score moved you from the grey bar into the band; it did **not** place a dot inside it. That is the whole module: a variant score is a strong **prior**, not a patient's **verdict.**

### Discussion (⚠️ safety debrief — vote, argue, re-vote)
- **Why does a *generative* genomic model carry dual-use risk that a purely *predictive* clinical model does not?** (Use Evo 2 as the example — its developers excluded eukaryote-infecting viruses from training and red-teamed outputs. What is the threat model for a model that can *write* DNA?)
- **You reclassify a VUS as likely-pathogenic using the model's score. What has to happen before that reaches a patient's chart?** (ACMG review, orthogonal evidence, functional data — what is the cost of a wrong 'pathogenic' call to a real family?)
- **Where does single-variant scoring break down *entirely* for predicting disease?** (Common, polygenic conditions — heart disease, diabetes. Why is a polygenic risk score a different modelling problem?)

### Stretch (fast finishers)
1. **Penetrance vs zygosity vs polygenic in one sentence each**, then defend in writing why the model's delta is a *prior*, not a verdict.
2. **Non-coding edge:** Evo 2 beats coding-only methods (like AlphaMissense) on **non-coding** and **indel** variants. Filter the BRCA1 table to a `consequence` that AlphaMissense couldn't score and argue why that frontier matters clinically (most of the genome — and most VUS — is non-coding).
3. **Context length:** HyenaDNA-small spans ~32 kb; Evo 2 reaches ~1 Mb. Name a variant class (distal enhancer, large structural variant) whose effect *only* the long-context model could see — and that the default model would miss.